# CS3807 Deep Learning Laboratory — Experiment 6
## End-to-End Study of RNN, LSTM and GRU for Sequence Learning and Video Understanding

This Colab notebook implements the complete assignment:

1. UCI-HAR raw inertial signals as **128 × 9** sequences.
2. 70/15/15 train/validation/test split from a balanced, manageable subset.
3. Temporal signal visualization.
4. Numerical RNN recurrence exercise.
5. Vanilla RNN, LSTM and GRU training.
6. Loss/accuracy curves and confusion matrices.
7. Accuracy, macro precision, macro recall, macro F1, parameters and training time.
8. Sequence-length experiment for 32, 64 and 128 time steps.
9. CNN → LSTM video-understanding pipeline using a small synthetic video dataset and frozen MobileNetV2.
10. Encoder-decoder LSTM sequence-to-sequence reversal task.
11. All required result tables.
12. Every plot is saved separately as **EPS, 600 DPI**.

**Important:** Numerical results are generated by execution and are not hard-coded.


In [1]:
# ============================================================
# 0. SETUP
# ============================================================
import os, sys, time, zipfile, shutil, urllib.request, random, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Keep execution reproducible and manageable in Colab.
USE_GPU = False
if not USE_GPU:
    try:
        tf.config.set_visible_devices([], "GPU")
    except Exception:
        pass

print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
print("CPU-only mode:", not USE_GPU)

# Assignment output directories
OUT = Path("/content/experiment6_outputs")
PLOTS = OUT / "plots_eps"
TABLES = OUT / "tables_csv"
MODELS = OUT / "models"

for p in [PLOTS, TABLES, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT)


TensorFlow: 2.20.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
CPU-only mode: True
Output directory: /content/experiment6_outputs


In [2]:
# ============================================================
# 1. ROBUST UCI-HAR DOWNLOAD + EXTRACTION
# ============================================================
ZIP_PATH = Path("/content/uci_har.zip")
EXTRACT_ROOT = Path("/content/UCI_HAR_DATASET")

UCI_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
]

def valid_uci_zip(path):
    if not path.exists() or not zipfile.is_zipfile(path):
        return False
    try:
        with zipfile.ZipFile(path) as z:
            names = z.namelist()
        required = [
            "y_train.txt", "y_test.txt",
            "body_acc_x_train.txt", "body_acc_y_train.txt", "body_acc_z_train.txt",
            "total_acc_x_train.txt", "total_acc_y_train.txt", "total_acc_z_train.txt",
            "body_gyro_x_train.txt", "body_gyro_y_train.txt", "body_gyro_z_train.txt"
        ]
        return all(any(n.endswith(r) for n in names) for r in required)
    except Exception:
        return False

if not valid_uci_zip(ZIP_PATH):
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    downloaded = False
    for url in UCI_URLS:
        try:
            print("Trying:", url)
            urllib.request.urlretrieve(url, ZIP_PATH)
            if valid_uci_zip(ZIP_PATH):
                downloaded = True
                break
        except Exception as e:
            print("Download attempt failed:", repr(e))
    if not downloaded:
        raise RuntimeError("Could not obtain a valid UCI-HAR ZIP.")
else:
    print("A valid UCI-HAR ZIP already exists.")

# Always extract into a fresh directory so stale/incomplete extraction cannot
# cause the previous y_train.txt / body_acc_x_train.txt error.
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)

print("Extraction completed:", EXTRACT_ROOT)

def find_required(filename):
    hits = list(EXTRACT_ROOT.rglob(filename))
    return str(hits[0]) if hits else None

required_files = {
    "y_train": "y_train.txt",
    "y_test": "y_test.txt",
    "body_acc_x_train": "body_acc_x_train.txt",
    "body_acc_y_train": "body_acc_y_train.txt",
    "body_acc_z_train": "body_acc_z_train.txt",
    "total_acc_x_train": "total_acc_x_train.txt",
    "total_acc_y_train": "total_acc_y_train.txt",
    "total_acc_z_train": "total_acc_z_train.txt",
    "body_gyro_x_train": "body_gyro_x_train.txt",
    "body_gyro_y_train": "body_gyro_y_train.txt",
    "body_gyro_z_train": "body_gyro_z_train.txt",
}

PATHS = {k: find_required(v) for k, v in required_files.items()}

print("\nDiscovered files:")
for k, v in PATHS.items():
    print(f"{k:22s} -> {v}")

missing = [k for k, v in PATHS.items() if v is None]
if missing:
    raise FileNotFoundError("Missing required UCI-HAR files: " + ", ".join(missing))

print("\nUCI-HAR dataset is ready.")


Trying: https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip
Trying: https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip
Extraction completed: /content/UCI_HAR_DATASET

Discovered files:
y_train                -> /content/UCI_HAR_DATASET/UCI HAR Dataset/train/y_train.txt
y_test                 -> /content/UCI_HAR_DATASET/UCI HAR Dataset/test/y_test.txt
body_acc_x_train       -> /content/UCI_HAR_DATASET/UCI HAR Dataset/train/Inertial Signals/body_acc_x_train.txt
body_acc_y_train       -> /content/UCI_HAR_DATASET/UCI HAR Dataset/train/Inertial Signals/body_acc_y_train.txt
body_acc_z_train       -> /content/UCI_HAR_DATASET/UCI HAR Dataset/train/Inertial Signals/body_acc_z_train.txt
total_acc_x_train      -> /content/UCI_HAR_DATASET/UCI HAR Dataset/train/Inertial Signals/total_acc_x_train.txt
total_acc_y_train      -> /content/UCI_HAR_DATASET/UCI HAR Dataset/train/Inertial Signals/total_acc_y_train.txt
total_acc_

In [3]:
# ============================================================
# 2. LOAD RAW 9-CHANNEL SIGNALS
# ============================================================
SIGNAL_NAMES = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z"
]

X_channels = []
for name in SIGNAL_NAMES:
    key = name + "_train"
    arr = np.loadtxt(PATHS[key])
    X_channels.append(arr)
    print(f"{name:15s}: {arr.shape}")

# Stack: samples × time × channels
X_full = np.stack(X_channels, axis=-1)

y_full = np.loadtxt(PATHS["y_train"]).astype(int) - 1

print("\nFull training partition:")
print("X:", X_full.shape)
print("y:", y_full.shape)
print("Classes:", np.unique(y_full))

CLASS_NAMES = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING"
]


body_acc_x     : (7352, 128)
body_acc_y     : (7352, 128)
body_acc_z     : (7352, 128)
total_acc_x    : (7352, 128)
total_acc_y    : (7352, 128)
total_acc_z    : (7352, 128)
body_gyro_x    : (7352, 128)
body_gyro_y    : (7352, 128)
body_gyro_z    : (7352, 128)

Full training partition:
X: (7352, 128, 9)
y: (7352,)
Classes: [0 1 2 3 4 5]


In [4]:
# ============================================================
# 3. SELECT A BALANCED MANAGEABLE SUBSET + 70/15/15 SPLIT
# ============================================================
# 400 windows per class = 2400 total windows.
# This is within the assignment's recommended 1500–3000 window range.
PER_CLASS = 400

selected_indices = []
for c in range(6):
    idx = np.where(y_full == c)[0]
    rng = np.random.default_rng(SEED + c)
    rng.shuffle(idx)
    selected_indices.extend(idx[:PER_CLASS])

selected_indices = np.array(selected_indices)
rng = np.random.default_rng(SEED)
rng.shuffle(selected_indices)

X = X_full[selected_indices]
y = y_full[selected_indices]

# First 70% train, remaining 30%; then split remaining equally into val/test.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Selected subset:", X.shape)
print("Training   :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Testing    :", X_test.shape, y_test.shape)

distribution = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Train": np.bincount(y_train, minlength=6),
    "Validation": np.bincount(y_val, minlength=6),
    "Test": np.bincount(y_test, minlength=6)
})
distribution["Total"] = distribution[["Train","Validation","Test"]].sum(axis=1)

print("\nClass distribution:")
display(distribution)
distribution.to_csv(TABLES / "table_class_distribution.csv", index=False)

shape_table = pd.DataFrame({
    "Set": ["Training", "Validation", "Testing"],
    "Samples": [len(X_train), len(X_val), len(X_test)],
    "Time steps": [X_train.shape[1]] * 3,
    "Features": [X_train.shape[2]] * 3
})
display(shape_table)
shape_table.to_csv(TABLES / "table_input_shapes.csv", index=False)


Selected subset: (2400, 128, 9)
Training   : (1680, 128, 9) (1680,)
Validation : (360, 128, 9) (360,)
Testing    : (360, 128, 9) (360,)

Class distribution:


,Class,Train,Validation,Test,Total
0,WALKING,280,60,60,400
1,WALKING_UPSTAIRS,280,60,60,400
2,WALKING_DOWNSTAIRS,280,60,60,400
3,SITTING,280,60,60,400
4,STANDING,280,60,60,400
5,LAYING,280,60,60,400


,Set,Samples,Time steps,Features
0,Training,1680,128,9
1,Validation,360,128,9
2,Testing,360,128,9


In [5]:
# ============================================================
# 4. TRAINING-ONLY NORMALIZATION
# ============================================================
# Fit scaler ONLY on training data.
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, X_train.shape[-1]))

def normalize_sequences(arr):
    shape = arr.shape
    return scaler.transform(arr.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)

X_train_n = normalize_sequences(X_train)
X_val_n = normalize_sequences(X_val)
X_test_n = normalize_sequences(X_test)

print("Normalized shapes:")
print(X_train_n.shape, X_val_n.shape, X_test_n.shape)

print("Training mean (approximately 0):", X_train_n.reshape(-1,9).mean(axis=0))
print("Training std  (approximately 1):", X_train_n.reshape(-1,9).std(axis=0))


Normalized shapes:
(1680, 128, 9) (360, 128, 9) (360, 128, 9)
Training mean (approximately 0): [ 3.34360567e-10 -2.45032084e-08  3.55196299e-08 -2.99617909e-07
  1.11624274e-07 -1.14824310e-07  4.13020906e-08 -1.52736384e-08
  9.71167147e-09]
Training std  (approximately 1): [0.9999051  0.99986047 0.9998645  0.9999866  0.99996835 0.9999581
 0.9998907  0.99988145 0.9998799 ]


In [6]:
# ============================================================
# 5. PLOT 1 — TEMPORAL SENSOR SIGNALS
# ============================================================
def save_eps(fig, filename):
    path = PLOTS / filename
    fig.savefig(path, format="eps", dpi=600, bbox_inches="tight")
    plt.close(fig)
    return path

# Representative examples from three different classes.
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

representatives = [0, 1, 3]  # Walking, Walking Upstairs, Sitting
channels = [0, 3, 6]         # body acc x, total acc x, body gyro x

for ax, cls, ch in zip(axes, representatives, channels):
    idx = np.where(y_train == cls)[0][0]
    ax.plot(np.arange(1,129), X_train_n[idx,:,ch], linewidth=1.2)
    ax.set_ylabel("Normalized value")
    ax.set_title(f"{CLASS_NAMES[cls]} — {SIGNAL_NAMES[ch]}")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Time step")
fig.suptitle("Plot 1: Representative Temporal Sensor Signals", fontsize=14)
fig.tight_layout(rect=[0,0,1,0.97])
save_eps(fig, "plot1_sensor_signals.eps")

print("Saved:", PLOTS / "plot1_sensor_signals.eps")


Saved: /content/experiment6_outputs/plots_eps/plot1_sensor_signals.eps


In [7]:
# ============================================================
# 6. NUMERICAL RNN EXERCISE
# ============================================================
x1, x2, x3 = 0.5, 0.7, 0.2
h = 0.0
Wx, Wh, b = 0.5, 0.8, 0.1

manual = []
for x in [x1, x2, x3]:
    h = np.tanh(Wx*x + Wh*h + b)
    manual.append(h)

numerical_table = pd.DataFrame({
    "Time step": [1,2,3],
    "x_t": [x1,x2,x3],
    "h_t": manual
})

display(numerical_table)
numerical_table.to_csv(TABLES / "table_numerical_rnn_exercise.csv", index=False)

print("h1 =", manual[0])
print("h2 =", manual[1])
print("h3 =", manual[2])


,Time step,x_t,h_t
0,1,0.5,0.336376
1,2,0.7,0.616352
2,3,0.2,0.599958


h1 = 0.3363755443363322
h2 = 0.6163517827505027
h3 = 0.5999579155496113


In [8]:
# ============================================================
# 7. MODEL BUILDERS
# ============================================================
NUM_CLASSES = 6
FEATURES = 9

BATCH_SIZE = 32
EPOCHS = 30

def make_model(kind="RNN", timesteps=128, units=32):
    model = keras.Sequential(name=f"{kind}_{timesteps}")
    model.add(layers.Input(shape=(timesteps, FEATURES)))

    if kind == "RNN":
        model.add(layers.SimpleRNN(units, activation="tanh"))
    elif kind == "LSTM":
        model.add(layers.LSTM(units))
    elif kind == "GRU":
        model.add(layers.GRU(units))
    else:
        raise ValueError("kind must be RNN, LSTM or GRU")

    model.add(layers.Dropout(0.20))
    model.add(layers.Dense(16, activation="relu"))
    model.add(layers.Dense(NUM_CLASSES, activation="softmax"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Verify architectures.
for kind in ["RNN", "LSTM", "GRU"]:
    m = make_model(kind)
    print("\n", kind)
    m.summary()



 RNN


Model: "RNN_128"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,974 (7.71 KB)

 Trainable params: 1,974 (7.71 KB)

...

Model: "GRU_128"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,758 (18.59 KB)

 Trainable params: 4,758 (18.59 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# ============================================================
# 8. TRAIN RNN / LSTM / GRU
# ============================================================
histories = {}
models = {}
training_times = {}

def train_model(kind):
    model = make_model(kind)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-5
        )
    ]

    start = time.perf_counter()

    history = model.fit(
        X_train_n, y_train,
        validation_data=(X_val_n, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    elapsed = time.perf_counter() - start

    models[kind] = model
    histories[kind] = history.history
    training_times[kind] = elapsed

    model.save(MODELS / f"{kind.lower()}_128.keras")

    print(f"{kind} training time: {elapsed:.2f} s")
    return model

for kind in ["RNN", "LSTM", "GRU"]:
    train_model(kind)


Epoch 1/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 5s 44ms/step - accuracy: 0.3435 - loss: 1.6562 - val_accuracy: 0.5167 - val_loss: 1.3476 - learning_rate: 0.0010
Epoch 2/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.5351 - loss: 1.2420 - val_accuracy: 0.5778 - val_loss: 1.0600 - learning_rate: 0.0010
Epoch 3/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5661 - loss: 1.0494 - val_accuracy: 0.6278 - val_loss: 0.9217 - learning_rate: 0.0010
Epoch 4/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6095 - loss: 0.9039 - val_accuracy: 0.6694 - val_loss: 0.7968 - learning_rate: 0.0010
Epoch 5/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6298 - loss: 0.8286 - val_accuracy: 0.6917 - val_loss: 0.7429 - learning_rate: 0.0010
Epoch 6/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6536 - loss: 0.7727 - val_accuracy: 0.7000 - val_loss: 0.6995 - learning_rate: 0.0010
Epoch 7/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6601 - loss: 0.7324 - val_acc

In [10]:
# ============================================================
# 9. PLOTS 2–7 — TRAINING/VALIDATION LOSS AND ACCURACY
# ============================================================
for kind in ["RNN", "LSTM", "GRU"]:
    h = histories[kind]
    epochs_run = range(1, len(h["loss"]) + 1)

    fig = plt.figure(figsize=(8,5))
    plt.plot(epochs_run, h["loss"], label="Training loss", linewidth=1.8)
    plt.plot(epochs_run, h["val_loss"], label="Validation loss", linewidth=1.8)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{kind}: Training and Validation Loss")
    plt.grid(alpha=0.25)
    plt.legend()
    fig.tight_layout()
    save_eps(fig, f"plot_{kind.lower()}_loss.eps")

    fig = plt.figure(figsize=(8,5))
    plt.plot(epochs_run, np.array(h["accuracy"])*100, label="Training accuracy", linewidth=1.8)
    plt.plot(epochs_run, np.array(h["val_accuracy"])*100, label="Validation accuracy", linewidth=1.8)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"{kind}: Training and Validation Accuracy")
    plt.grid(alpha=0.25)
    plt.legend()
    fig.tight_layout()
    save_eps(fig, f"plot_{kind.lower()}_accuracy.eps")

print("Saved six training plots.")


Saved six training plots.


In [11]:
# ============================================================
# 10. TEST METRICS + CONFUSION MATRICES
# ============================================================
metrics_rows = []
predictions = {}

for kind, model in models.items():
    probs = model.predict(X_test_n, batch_size=128, verbose=0)
    pred = np.argmax(probs, axis=1)
    predictions[kind] = pred

    metrics_rows.append({
        "Model": kind,
        "Accuracy (%)": accuracy_score(y_test, pred) * 100,
        "Macro Precision (%)": precision_score(y_test, pred, average="macro", zero_division=0) * 100,
        "Macro Recall (%)": recall_score(y_test, pred, average="macro", zero_division=0) * 100,
        "Macro F1 (%)": f1_score(y_test, pred, average="macro", zero_division=0) * 100,
        "Parameters": model.count_params(),
        "Training Time (s)": training_times[kind]
    })

    cm = confusion_matrix(y_test, pred)

    fig = plt.figure(figsize=(8,7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
    )
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title(f"{kind} Confusion Matrix")
    plt.tight_layout()
    save_eps(fig, f"plot_{kind.lower()}_confusion_matrix.eps")

metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df.round(4))
metrics_df.to_csv(TABLES / "table_rnn_lstm_gru_metrics.csv", index=False)


,Model,Accuracy (%),Macro Precision (%),Macro Recall (%),Macro F1 (%),Parameters,Training Time (s)
0,RNN,73.8889,73.5316,73.8889,73.6621,1974,28.8654
1,LSTM,95.5556,95.5636,95.5556,95.5552,6006,50.7701
2,GRU,94.4444,94.5685,94.4444,94.4383,4758,57.5632


In [12]:
# ============================================================
# 11. PLOT 8 — MODEL PERFORMANCE COMPARISON
# ============================================================
plot_df = metrics_df.set_index("Model")[[
    "Accuracy (%)", "Macro Precision (%)", "Macro Recall (%)", "Macro F1 (%)"
]]

fig = plt.figure(figsize=(10,6))
plot_df.plot(kind="bar", ax=plt.gca())
plt.ylabel("Score (%)")
plt.xlabel("Model")
plt.title("Plot 8: RNN vs LSTM vs GRU Test Performance")
plt.xticks(rotation=0)
plt.ylim(0,100)
plt.grid(axis="y", alpha=0.25)
plt.legend(loc="lower right")
fig.tight_layout()
save_eps(fig, "plot8_model_performance.eps")

print("Saved:", PLOTS / "plot8_model_performance.eps")


Saved: /content/experiment6_outputs/plots_eps/plot8_model_performance.eps


In [13]:
# ============================================================
# 12. ARCHITECTURE / COMPLEXITY COMPARISON TABLE
# ============================================================
architecture_df = pd.DataFrame({
    "Property": [
        "Hidden state", "Cell state", "Forget gate", "Input gate",
        "Output gate", "Update gate", "Reset gate",
        "Parameters", "Training Time (s)", "Test Macro F1 (%)"
    ],
    "RNN": [
        "Yes", "No", "No", "No", "No", "No", "No",
        int(metrics_df.loc[metrics_df.Model=="RNN","Parameters"].iloc[0]),
        metrics_df.loc[metrics_df.Model=="RNN","Training Time (s)"].iloc[0],
        metrics_df.loc[metrics_df.Model=="RNN","Macro F1 (%)"].iloc[0]
    ],
    "LSTM": [
        "Yes", "Yes", "Yes", "Yes", "Yes", "No", "No",
        int(metrics_df.loc[metrics_df.Model=="LSTM","Parameters"].iloc[0]),
        metrics_df.loc[metrics_df.Model=="LSTM","Training Time (s)"].iloc[0],
        metrics_df.loc[metrics_df.Model=="LSTM","Macro F1 (%)"].iloc[0]
    ],
    "GRU": [
        "Yes", "No", "No", "No", "No", "Yes", "Yes",
        int(metrics_df.loc[metrics_df.Model=="GRU","Parameters"].iloc[0]),
        metrics_df.loc[metrics_df.Model=="GRU","Training Time (s)"].iloc[0],
        metrics_df.loc[metrics_df.Model=="GRU","Macro F1 (%)"].iloc[0]
    ]
})

display(architecture_df)
architecture_df.to_csv(TABLES / "table_architecture_comparison.csv", index=False)


,Property,RNN,LSTM,GRU
0,Hidden state,Yes,Yes,Yes
1,Cell state,No,Yes,No
2,Forget gate,No,Yes,No
3,Input gate,No,Yes,No
4,Output gate,No,Yes,No
5,Update gate,No,No,Yes
6,Reset gate,No,No,Yes
7,Parameters,1974,6006,4758
8,Training Time (s),28.865422,50.77006,57.563243
9,Test Macro F1 (%),73.662083,95.555247,94.438265


In [14]:
# ============================================================
# 13. SEQUENCE LENGTH EXPERIMENT: 32, 64, 128
# ============================================================
SEQ_LENGTHS = [32, 64, 128]
seq_results = []

# Use a lighter epoch budget for the controlled ablation.
SEQ_EPOCHS = 15

def train_sequence_length(kind, T):
    model = make_model(kind, timesteps=T)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
        )
    ]

    start = time.perf_counter()
    model.fit(
        X_train_n[:, :T, :], y_train,
        validation_data=(X_val_n[:, :T, :], y_val),
        epochs=SEQ_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=0
    )
    elapsed = time.perf_counter() - start

    pred = np.argmax(
        model.predict(X_test_n[:, :T, :], batch_size=128, verbose=0), axis=1
    )

    return (
        f1_score(y_test, pred, average="macro", zero_division=0)*100,
        accuracy_score(y_test, pred)*100,
        model.count_params(),
        elapsed
    )

for T in SEQ_LENGTHS:
    for kind in ["RNN", "LSTM", "GRU"]:
        f1v, accv, params, elapsed = train_sequence_length(kind, T)
        seq_results.append({
            "Sequence Length": T,
            "Model": kind,
            "Accuracy (%)": accv,
            "Macro F1 (%)": f1v,
            "Parameters": params,
            "Training Time (s)": elapsed
        })
        print(T, kind, "F1:", round(f1v,3))

seq_df = pd.DataFrame(seq_results)
display(seq_df.round(4))
seq_df.to_csv(TABLES / "table_sequence_length_results.csv", index=False)


32 RNN F1: 75.088
32 LSTM F1: 91.076
32 GRU F1: 91.356
64 RNN F1: 70.157
64 LSTM F1: 86.227
64 GRU F1: 93.904
128 RNN F1: 69.687
128 LSTM F1: 69.951
128 GRU F1: 95.016


,Sequence Length,Model,Accuracy (%),Macro F1 (%),Parameters,Training Time (s)
0,32,RNN,75.0000,75.0878,1974,6.4168
1,32,LSTM,91.1111,91.0762,6006,10.4290
2,32,GRU,91.3889,91.3565,4758,11.0687
3,64,RNN,70.0000,70.1574,1974,7.5730
4,64,LSTM,86.3889,86.2270,6006,16.4645
5,64,GRU,93.8889,93.9041,4758,17.8989
6,128,RNN,70.5556,69.6870,1974,13.1722
7,128,LSTM,70.2778,69.9507,6006,14.6756
8,128,GRU,95.0000,95.0155,4758,32.9439


In [15]:
# ============================================================
# 14. PLOT 9 — SEQUENCE LENGTH VS MACRO F1
# ============================================================
fig = plt.figure(figsize=(9,6))

for kind in ["RNN", "LSTM", "GRU"]:
    d = seq_df[seq_df["Model"] == kind]
    plt.plot(
        d["Sequence Length"], d["Macro F1 (%)"],
        marker="o", linewidth=1.8, label=kind
    )

plt.xlabel("Sequence length")
plt.ylabel("Macro F1 (%)")
plt.title("Plot 9: Sequence Length vs Test Macro F1")
plt.xticks(SEQ_LENGTHS)
plt.grid(alpha=0.25)
plt.legend()
fig.tight_layout()
save_eps(fig, "plot9_sequence_length_vs_f1.eps")


PosixPath('/content/experiment6_outputs/plots_eps/plot9_sequence_length_vs_f1.eps')

In [16]:
# ============================================================
# 15. VIDEO DATASET — SMALL REPRODUCIBLE SYNTHETIC ACTION VIDEOS
# ============================================================
# The assignment recommends a small UCF101 subset. Downloading UCF101 in full
# is several GB and is unnecessary for demonstrating the CNN->LSTM pipeline.
# This notebook therefore creates a compact video-like benchmark locally:
# 4 action classes represented by different moving geometric patterns.
#
# Each "video" has 10 RGB frames of 64x64 pixels.
# MobileNetV2 is then used as the frozen pretrained CNN feature extractor.

VIDEO_CLASSES = ["HorizontalMove", "VerticalMove", "DiagonalMove", "Pulse"]
VIDEO_FRAMES = 10
VIDEO_SIZE = 64
VIDEOS_PER_CLASS = 36

def make_frame(action, t, n=64):
    img = np.zeros((n,n,3), dtype=np.float32)

    # Coordinates vary with time.
    u = int(8 + (n-16) * t / (VIDEO_FRAMES-1))
    v = int(8 + (n-16) * (1-t))

    yy, xx = np.mgrid[0:n,0:n]

    if action == "HorizontalMove":
        mask = (xx-u)**2 + (yy-n//2)**2 < 9**2
        img[mask] = [1.0, 0.25, 0.1]

    elif action == "VerticalMove":
        mask = (xx-n//2)**2 + (yy-u)**2 < 9**2
        img[mask] = [0.1, 0.8, 0.25]

    elif action == "DiagonalMove":
        mask = (xx-u)**2 + (yy-u)**2 < 9**2
        img[mask] = [0.15, 0.4, 1.0]

    else:
        # Pulsing square/circle size
        r = int(4 + 10*(0.5+0.5*np.sin(2*np.pi*t)))
        mask = (xx-n//2)**2 + (yy-n//2)**2 < r**2
        img[mask] = [0.8, 0.2, 0.8]

    # Add small fixed background texture.
    noise = np.random.default_rng(SEED + int(t*1000)).normal(0, 0.015, img.shape)
    img = np.clip(img + noise, 0, 1)

    return img

video_X = []
video_y = []

for c, action in enumerate(VIDEO_CLASSES):
    for v_id in range(VIDEOS_PER_CLASS):
        frames = []
        for t in range(VIDEO_FRAMES):
            frames.append(make_frame(action, t))
        video_X.append(frames)
        video_y.append(c)

video_X = np.array(video_X, dtype=np.float32)
video_y = np.array(video_y, dtype=np.int32)

print("Video tensor:", video_X.shape)
print("Video labels:", video_y.shape)


Video tensor: (144, 10, 64, 64, 3)
Video labels: (144,)


In [17]:
# ============================================================
# 16. PLOT 10 — VIDEO SAMPLE FRAMES
# ============================================================
fig, axes = plt.subplots(1, VIDEO_FRAMES, figsize=(18,3))

for t, ax in enumerate(axes):
    ax.imshow(video_X[0,t])
    ax.set_title(f"Frame {t+1}")
    ax.axis("off")

fig.suptitle("Plot 10: Sample Video Frames — " + VIDEO_CLASSES[video_y[0]])
fig.tight_layout()
save_eps(fig, "plot10_video_sample_frames.eps")


PosixPath('/content/experiment6_outputs/plots_eps/plot10_video_sample_frames.eps')

In [18]:
# ============================================================
# 17. VIDEO TRAIN/VALIDATION/TEST SPLIT
# ============================================================
vX_train, vX_temp, vy_train, vy_temp = train_test_split(
    video_X, video_y, test_size=0.30,
    random_state=SEED, stratify=video_y
)

vX_val, vX_test, vy_val, vy_test = train_test_split(
    vX_temp, vy_temp, test_size=0.50,
    random_state=SEED, stratify=vy_temp
)

print("Video train:", vX_train.shape)
print("Video val  :", vX_val.shape)
print("Video test :", vX_test.shape)


Video train: (100, 10, 64, 64, 3)
Video val  : (22, 10, 64, 64, 3)
Video test : (22, 10, 64, 64, 3)


In [19]:
# ============================================================
# 18. FROZEN MOBILENETV2 CNN FEATURE EXTRACTION
# ============================================================
# MobileNetV2 with ImageNet weights, classification head removed.
cnn = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    pooling="avg",
    input_shape=(224,224,3)
)
cnn.trainable = False

FEATURE_DIM = cnn.output_shape[-1]
print("CNN feature dimension:", FEATURE_DIM)

def extract_video_features(videos, batch_size=32):
    n, t, h, w, c = videos.shape
    flat = videos.reshape(n*t, h, w, c)
    flat = tf.image.resize(flat, (224,224))
    flat = tf.keras.applications.mobilenet_v2.preprocess_input(flat * 255.0)

    features = cnn.predict(flat, batch_size=batch_size, verbose=0)
    return features.reshape(n, t, -1).astype(np.float32)

print("Extracting training features...")
vF_train = extract_video_features(vX_train)

print("Extracting validation features...")
vF_val = extract_video_features(vX_val)

print("Extracting test features...")
vF_test = extract_video_features(vX_test)

print("CNN -> LSTM input:")
print("Training:", vF_train.shape)
print("Validation:", vF_val.shape)
print("Testing:", vF_test.shape)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
CNN feature dimension: 1280
Extracting training features...
Extracting validation features...
Extracting test features...
CNN -> LSTM input:
Training: (100, 10, 1280)
Validation: (22, 10, 1280)
Testing: (22, 10, 1280)


In [20]:
# ============================================================
# 19. CNN-LSTM VIDEO CLASSIFIER
# ============================================================
video_model = keras.Sequential([
    layers.Input(shape=(VIDEO_FRAMES, FEATURE_DIM)),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(len(VIDEO_CLASSES), activation="softmax")
])

video_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

video_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
    )
]

start = time.perf_counter()

video_history = video_model.fit(
    vF_train, vy_train,
    validation_data=(vF_val, vy_val),
    epochs=20,
    batch_size=8,
    callbacks=video_callbacks,
    verbose=1
)

video_time = time.perf_counter() - start

video_probs = video_model.predict(vF_test, verbose=0)
video_pred = np.argmax(video_probs, axis=1)

video_metrics = {
    "Model": "CNN-LSTM",
    "Accuracy (%)": accuracy_score(vy_test, video_pred)*100,
    "Macro Precision (%)": precision_score(vy_test, video_pred, average="macro", zero_division=0)*100,
    "Macro Recall (%)": recall_score(vy_test, video_pred, average="macro", zero_division=0)*100,
    "Macro F1 (%)": f1_score(vy_test, video_pred, average="macro", zero_division=0)*100,
    "Parameters": video_model.count_params(),
    "Training Time (s)": video_time
}

video_metrics_df = pd.DataFrame([video_metrics])
display(video_metrics_df.round(4))
video_metrics_df.to_csv(TABLES / "table_cnn_lstm_video_metrics.csv", index=False)

video_model.save(MODELS / "cnn_lstm_video.keras")


Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.5700 - loss: 1.1050 - val_accuracy: 0.7727 - val_loss: 0.8000 - learning_rate: 0.0010
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9400 - loss: 0.6189 - val_accuracy: 1.0000 - val_loss: 0.3919 - learning_rate: 0.0010
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.3535 - val_accuracy: 1.0000 - val_loss: 0.2509 - learning_rate: 0.0010
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.2386 - val_accuracy: 1.0000 - val_loss: 0.1454 - learning_rate: 0.0010
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.1346 - val_accuracy: 1.0000 - val_loss: 0.0959 - learning_rate: 0.0010
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.1109 - val_accuracy: 1.0000 - val_loss: 0.0694 - learning_rate: 0.0010
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0773 - val_acc

,Model,Accuracy (%),Macro Precision (%),Macro Recall (%),Macro F1 (%),Parameters,Training Time (s)
0,CNN-LSTM,100.0,100.0,100.0,100.0,168660,5.9861


In [21]:
# ============================================================
# 20. PLOTS 11–12 — VIDEO TRAINING CURVES
# ============================================================
vh = video_history.history
ep = range(1, len(vh["loss"])+1)

fig = plt.figure(figsize=(8,5))
plt.plot(ep, vh["loss"], label="Training loss", linewidth=1.8)
plt.plot(ep, vh["val_loss"], label="Validation loss", linewidth=1.8)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Plot 11: CNN-LSTM Video Training/Validation Loss")
plt.grid(alpha=0.25)
plt.legend()
fig.tight_layout()
save_eps(fig, "plot11_video_loss.eps")

fig = plt.figure(figsize=(8,5))
plt.plot(ep, np.array(vh["accuracy"])*100, label="Training accuracy", linewidth=1.8)
plt.plot(ep, np.array(vh["val_accuracy"])*100, label="Validation accuracy", linewidth=1.8)
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Plot 12: CNN-LSTM Video Training/Validation Accuracy")
plt.grid(alpha=0.25)
plt.legend()
fig.tight_layout()
save_eps(fig, "plot12_video_accuracy.eps")


PosixPath('/content/experiment6_outputs/plots_eps/plot12_video_accuracy.eps')

In [22]:
# ============================================================
# 21. PLOT 13 — VIDEO CONFUSION MATRIX
# ============================================================
cm_video = confusion_matrix(vy_test, video_pred)

fig = plt.figure(figsize=(7,6))
sns.heatmap(
    cm_video, annot=True, fmt="d", cmap="Blues",
    xticklabels=VIDEO_CLASSES, yticklabels=VIDEO_CLASSES
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Plot 13: CNN-LSTM Video Confusion Matrix")
plt.tight_layout()
save_eps(fig, "plot13_video_confusion_matrix.eps")

# Five actual predictions with confidence.
example_rows = []
for i in range(min(5, len(vy_test))):
    example_rows.append({
        "Sample": i+1,
        "Actual": VIDEO_CLASSES[vy_test[i]],
        "Predicted": VIDEO_CLASSES[video_pred[i]],
        "Confidence (%)": float(np.max(video_probs[i])*100)
    })

video_examples_df = pd.DataFrame(example_rows)
display(video_examples_df.round(3))
video_examples_df.to_csv(TABLES / "table_video_examples.csv", index=False)


,Sample,Actual,Predicted,Confidence (%)
0,1,HorizontalMove,HorizontalMove,99.108
1,2,HorizontalMove,HorizontalMove,99.108
2,3,Pulse,Pulse,99.275
3,4,DiagonalMove,DiagonalMove,99.444
4,5,Pulse,Pulse,99.275


In [23]:
# ============================================================
# 22. SEQUENCE-TO-SEQUENCE REVERSAL DATASET
# ============================================================
VOCAB_SIZE = 10
SEQ2SEQ_LEN = 5
N_SEQ2SEQ = 3000

rng = np.random.default_rng(SEED)

seq_inputs = rng.integers(
    1, VOCAB_SIZE, size=(N_SEQ2SEQ, SEQ2SEQ_LEN)
)
seq_targets = np.flip(seq_inputs, axis=1).copy()

# Decoder input uses a START token = 0.
decoder_inputs = np.concatenate([
    np.zeros((N_SEQ2SEQ,1), dtype=np.int32),
    seq_targets[:, :-1]
], axis=1)

sX_train, sX_temp, sY_train, sY_temp, sD_train, sD_temp = train_test_split(
    seq_inputs, seq_targets, decoder_inputs,
    test_size=0.30, random_state=SEED
)

sX_val, sX_test, sY_val, sY_test, sD_val, sD_test = train_test_split(
    sX_temp, sY_temp, sD_temp,
    test_size=0.50, random_state=SEED
)

print("Seq2seq train:", sX_train.shape)
print("Seq2seq val  :", sX_val.shape)
print("Seq2seq test :", sX_test.shape)


Seq2seq train: (2100, 5)
Seq2seq val  : (450, 5)
Seq2seq test : (450, 5)


In [24]:
# ============================================================
# 23. ENCODER-DECODER LSTM
# ============================================================
EMBED_DIM = 32
LATENT_DIM = 64

# Encoder
encoder_inputs = keras.Input(shape=(SEQ2SEQ_LEN,), dtype="int32")
enc_emb = layers.Embedding(VOCAB_SIZE, EMBED_DIM)(encoder_inputs)
encoder_lstm = layers.LSTM(LATENT_DIM, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

# Decoder
decoder_inputs_keras = keras.Input(shape=(SEQ2SEQ_LEN,), dtype="int32")
dec_emb_layer = layers.Embedding(VOCAB_SIZE, EMBED_DIM)
dec_emb = dec_emb_layer(decoder_inputs_keras)

decoder_lstm = layers.LSTM(
    LATENT_DIM, return_sequences=True, return_state=True
)
decoder_outputs, _, _ = decoder_lstm(
    dec_emb, initial_state=[state_h, state_c]
)

decoder_dense = layers.Dense(VOCAB_SIZE, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

seq2seq_model = keras.Model(
    [encoder_inputs, decoder_inputs_keras],
    decoder_outputs
)

seq2seq_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

seq2seq_model.summary()

start = time.perf_counter()

seq_history = seq2seq_model.fit(
    [sX_train, sD_train],
    sY_train[..., None],
    validation_data=([sX_val, sD_val], sY_val[..., None]),
    epochs=25,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
        )
    ],
    verbose=1
)

seq_time = time.perf_counter() - start


Model: "functional_61"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_17      │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_18      │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 5, 32)     │        320 │ input_layer_17[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 5, 32)     │        320 │ input_layer_18[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 64),      │     24,832 │ embedding[0][0]   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ [(None, 5, 64),   │     24,832 │ embedding_1[0][0… │
│                     │ (None, 64),       │            │ lstm_6[0][1],     │
│                     │ (None, 64)]       │            │ lstm_6[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_32 (Dense)    │ (None, 5, 10)     │        650 │ lstm_7[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 50,954 (199.04 KB)

 Trainable params: 50,954 (199.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.1951 - loss: 2.2384 - val_accuracy: 0.2200 - val_loss: 2.1400 - learning_rate: 0.0010
Epoch 2/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3195 - loss: 1.9010 - val_accuracy: 0.3560 - val_loss: 1.7159 - learning_rate: 0.0010
Epoch 3/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3912 - loss: 1.5864 - val_accuracy: 0.4156 - val_loss: 1.4818 - learning_rate: 0.0010
Epoch 4/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4947 - loss: 1.3005 - val_accuracy: 0.5844 - val_loss: 1.1072 - learning_rate: 0.0010
Epoch 5/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7095 - loss: 0.8800 - val_accuracy: 0.8298 - val_loss: 0.6757 - learning_rate: 0.0010
Epoch 6/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8765 - loss: 0.5456 - val_accuracy: 0.9244 - val_loss: 0.4284 - learning_rate: 0.0010
Epoch 7/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9449 - loss: 0.3595 - val_accuracy:

In [25]:
# ============================================================
# 24. PLOT 14 — SEQ2SEQ TRAINING/VALIDATION LOSS
# ============================================================
sh = seq_history.history
ep = range(1, len(sh["loss"])+1)

fig = plt.figure(figsize=(8,5))
plt.plot(ep, sh["loss"], label="Training loss", linewidth=1.8)
plt.plot(ep, sh["val_loss"], label="Validation loss", linewidth=1.8)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Plot 14: Seq2Seq Training and Validation Loss")
plt.grid(alpha=0.25)
plt.legend()
fig.tight_layout()
save_eps(fig, "plot14_seq2seq_loss.eps")


PosixPath('/content/experiment6_outputs/plots_eps/plot14_seq2seq_loss.eps')

In [26]:
# ============================================================
# 25. SEQ2SEQ EVALUATION
# ============================================================
seq_probs = seq2seq_model.predict(
    [sX_test, sD_test], batch_size=128, verbose=0
)

seq_pred = np.argmax(seq_probs, axis=-1)

token_accuracy = np.mean(seq_pred == sY_test)

sequence_accuracy = np.mean(
    np.all(seq_pred == sY_test, axis=1)
)

seq_metrics_df = pd.DataFrame([{
    "Task": "Sequence Reversal",
    "Token Accuracy (%)": token_accuracy*100,
    "Sequence Accuracy (%)": sequence_accuracy*100,
    "Training Time (s)": seq_time,
    "Parameters": seq2seq_model.count_params()
}])

display(seq_metrics_df.round(4))
seq_metrics_df.to_csv(TABLES / "table_seq2seq_metrics.csv", index=False)

# Five required examples
example_rows = []
for i in range(5):
    example_rows.append({
        "Sample": i+1,
        "Input Sequence": " ".join(map(str, sX_test[i])),
        "Expected Output": " ".join(map(str, sY_test[i])),
        "Predicted Output": " ".join(map(str, seq_pred[i])),
        "Correct Sequence": bool(np.all(seq_pred[i] == sY_test[i]))
    })

seq_examples_df = pd.DataFrame(example_rows)
display(seq_examples_df)
seq_examples_df.to_csv(TABLES / "table_seq2seq_examples.csv", index=False)


,Task,Token Accuracy (%),Sequence Accuracy (%),Training Time (s),Parameters
0,Sequence Reversal,99.9111,99.5556,13.4616,50954


,Sample,Input Sequence,Expected Output,Predicted Output,Correct Sequence
0,1,6 2 5 4 8,8 4 5 2 6,8 4 5 2 6,True
1,2,4 6 7 8 2,2 8 7 6 4,2 8 7 6 4,True
2,3,1 8 5 6 1,1 6 5 8 1,1 6 5 8 1,True
3,4,8 8 9 9 8,8 9 9 8 8,8 9 9 8 8,True
4,5,8 6 3 4 3,3 4 3 6 8,3 4 3 6 8,True


In [27]:
# ============================================================
# 26. CONSOLIDATED RESULTS TABLE
# ============================================================
main_consolidated = metrics_df.copy()

video_row = pd.DataFrame([{
    "Model": "CNN-LSTM",
    "Accuracy (%)": video_metrics["Accuracy (%)"],
    "Macro Precision (%)": video_metrics["Macro Precision (%)"],
    "Macro Recall (%)": video_metrics["Macro Recall (%)"],
    "Macro F1 (%)": video_metrics["Macro F1 (%)"],
    "Parameters": video_metrics["Parameters"],
    "Training Time (s)": video_metrics["Training Time (s)"]
}])

consolidated = pd.concat([main_consolidated, video_row], ignore_index=True)

display(consolidated.round(4))
consolidated.to_csv(TABLES / "table_consolidated_results.csv", index=False)

# Also produce a clean assignment-style table.
assignment_table = consolidated[
    ["Model", "Accuracy (%)", "Macro Precision (%)",
     "Macro Recall (%)", "Macro F1 (%)", "Parameters"]
].copy()

display(assignment_table.round(4))
assignment_table.to_csv(TABLES / "table_assignment_summary.csv", index=False)


,Model,Accuracy (%),Macro Precision (%),Macro Recall (%),Macro F1 (%),Parameters,Training Time (s)
0,RNN,73.8889,73.5316,73.8889,73.6621,1974,28.8654
1,LSTM,95.5556,95.5636,95.5556,95.5552,6006,50.7701
2,GRU,94.4444,94.5685,94.4444,94.4383,4758,57.5632
3,CNN-LSTM,100.0000,100.0000,100.0000,100.0000,168660,5.9861


,Model,Accuracy (%),Macro Precision (%),Macro Recall (%),Macro F1 (%),Parameters
0,RNN,73.8889,73.5316,73.8889,73.6621,1974
1,LSTM,95.5556,95.5636,95.5556,95.5552,6006
2,GRU,94.4444,94.5685,94.4444,94.4383,4758
3,CNN-LSTM,100.0000,100.0000,100.0000,100.0000,168660


In [28]:
# ============================================================
# 27. AUTOMATIC INFERENCE SUMMARY FROM OBSERVED RESULTS
# ============================================================
def best_model(metric):
    row = metrics_df.loc[metrics_df[metric].idxmax()]
    return row["Model"], row[metric]

best_acc_model, best_acc = best_model("Accuracy (%)")
best_f1_model, best_f1 = best_model("Macro F1 (%)")

print("Observed result summary")
print("-----------------------")
print(f"Highest test accuracy in this run: {best_acc_model} ({best_acc:.2f}%)")
print(f"Highest test macro F1 in this run: {best_f1_model} ({best_f1:.2f}%)")

for kind in ["RNN", "LSTM", "GRU"]:
    h = histories[kind]
    train_final = h["accuracy"][-1] * 100
    val_final = h["val_accuracy"][-1] * 100
    gap = train_final - val_final
    print(f"{kind}: final training accuracy={train_final:.2f}%, "
          f"validation accuracy={val_final:.2f}%, gap={gap:.2f} percentage points")

# Activity-wise report for the best-F1 recurrent model.
best_kind = best_f1_model
print(f"\nClassification report for {best_kind}:")
print(classification_report(
    y_test, predictions[best_kind],
    target_names=CLASS_NAMES, digits=4, zero_division=0
))


Observed result summary
-----------------------
Highest test accuracy in this run: LSTM (95.56%)
Highest test macro F1 in this run: LSTM (95.56%)
RNN: final training accuracy=78.75%, validation accuracy=83.33%, gap=-4.58 percentage points
LSTM: final training accuracy=96.37%, validation accuracy=96.67%, gap=-0.30 percentage points
GRU: final training accuracy=95.60%, validation accuracy=96.39%, gap=-0.79 percentage points

Classification report for LSTM:
                    precision    recall  f1-score   support

           WALKING     0.9672    0.9833    0.9752        60
  WALKING_UPSTAIRS     0.9831    0.9667    0.9748        60
WALKING_DOWNSTAIRS     1.0000    1.0000    1.0000        60
           SITTING     0.8852    0.9000    0.8926        60
          STANDING     0.8983    0.8833    0.8908        60
            LAYING     1.0000    1.0000    1.0000        60

          accuracy                         0.9556       360
         macro avg     0.9556    0.9556    0.9556       360

In [29]:
# ============================================================
# 28. PACKAGE ALL TABLES, PLOTS AND MODELS
# ============================================================
import glob

# Save a plain-text list of generated artifacts.
artifact_rows = []

for p in sorted(PLOTS.glob("*.eps")):
    artifact_rows.append({"Type":"EPS Plot", "File":str(p), "Size (KB)":p.stat().st_size/1024})

for p in sorted(TABLES.glob("*.csv")):
    artifact_rows.append({"Type":"CSV Table", "File":str(p), "Size (KB)":p.stat().st_size/1024})

for p in sorted(MODELS.glob("*.keras")):
    artifact_rows.append({"Type":"Keras Model", "File":str(p), "Size (KB)":p.stat().st_size/1024})

artifacts_df = pd.DataFrame(artifact_rows)
display(artifacts_df)
artifacts_df.to_csv(OUT / "artifact_manifest.csv", index=False)

ZIP_OUTPUT = "/content/Experiment6_outputs.zip"
if os.path.exists(ZIP_OUTPUT):
    os.remove(ZIP_OUTPUT)

shutil.make_archive(
    "/content/Experiment6_outputs",
    "zip",
    OUT
)

print("\nCOMPLETE")
print("Plots:", PLOTS)
print("Tables:", TABLES)
print("Models:", MODELS)
print("ZIP:", ZIP_OUTPUT)

print("\nGenerated EPS plots:")
for p in sorted(PLOTS.glob("*.eps")):
    print(" ", p.name)


,Type,File,Size (KB)
0,EPS Plot,/content/experiment6_outputs/plots_eps/plot10_...,56854.572266
1,EPS Plot,/content/experiment6_outputs/plots_eps/plot11_...,25.610352
2,EPS Plot,/content/experiment6_outputs/plots_eps/plot12_...,27.665039
3,EPS Plot,/content/experiment6_outputs/plots_eps/plot13_...,2725.528320
4,EPS Plot,/content/experiment6_outputs/plots_eps/plot14_...,21.661133
5,EPS Plot,/content/experiment6_outputs/plots_eps/plot1_s...,46.141602
6,EPS Plot,/content/experiment6_outputs/plots_eps/plot8_m...,26.234375
7,EPS Plot,/content/experiment6_outputs/plots_eps/plot9_s...,29.314453
8,EPS Plot,/content/experiment6_outputs/plots_eps/plot_gr...,27.004883
9,EPS Plot,/content/experiment6_outputs/plots_eps/plot_gr...,2220.268555



COMPLETE
Plots: /content/experiment6_outputs/plots_eps
Tables: /content/experiment6_outputs/tables_csv
Models: /content/experiment6_outputs/models
ZIP: /content/Experiment6_outputs.zip

Generated EPS plots:
  plot10_video_sample_frames.eps
  plot11_video_loss.eps
  plot12_video_accuracy.eps
  plot13_video_confusion_matrix.eps
  plot14_seq2seq_loss.eps
  plot1_sensor_signals.eps
  plot8_model_performance.eps
  plot9_sequence_length_vs_f1.eps
  plot_gru_accuracy.eps
  plot_gru_confusion_matrix.eps
  plot_gru_loss.eps
  plot_lstm_accuracy.eps
  plot_lstm_confusion_matrix.eps
  plot_lstm_loss.eps
  plot_rnn_accuracy.eps
  plot_rnn_confusion_matrix.eps
  plot_rnn_loss.eps


In [32]:
import shutil

shutil.make_archive(
    "/content/experiment6_outputs/plots_eps",
    "zip",
    "/content/plots_eps"
)

print("ZIP created: /content/Lab6_Plots.zip")

FileNotFoundError: [Errno 2] No such file or directory: '/content/plots_eps'

In [33]:
!zip -r plots_eps.zip experiment6_outputs/plots_eps


  adding: experiment6_outputs/plots_eps/ (stored 0%)
  adding: experiment6_outputs/plots_eps/plot_rnn_confusion_matrix.eps (deflated 99%)
  adding: experiment6_outputs/plots_eps/plot_lstm_accuracy.eps (deflated 68%)
  adding: experiment6_outputs/plots_eps/plot11_video_loss.eps (deflated 69%)
  adding: experiment6_outputs/plots_eps/plot8_model_performance.eps (deflated 68%)
  adding: experiment6_outputs/plots_eps/plot1_sensor_signals.eps (deflated 72%)
  adding: experiment6_outputs/plots_eps/plot_rnn_accuracy.eps (deflated 68%)
  adding: experiment6_outputs/plots_eps/plot_lstm_loss.eps (deflated 70%)
  adding: experiment6_outputs/plots_eps/plot14_seq2seq_loss.eps (deflated 69%)
  adding: experiment6_outputs/plots_eps/plot12_video_accuracy.eps (deflated 69%)
  adding: experiment6_outputs/plots_eps/plot_gru_confusion_matrix.eps (deflated 99%)
  adding: experiment6_outputs/plots_eps/plot_lstm_confusion_matrix.eps (deflated 99%)
  adding: experiment6_outputs/plots_eps/plot_gru_loss.eps (def

## Final submission checklist

After the notebook finishes:

- `plot1_sensor_signals.eps` — temporal sensor visualization
- `plot_rnn_loss.eps`, `plot_rnn_accuracy.eps`
- `plot_lstm_loss.eps`, `plot_lstm_accuracy.eps`
- `plot_gru_loss.eps`, `plot_gru_accuracy.eps`
- three recurrent confusion matrices
- `plot8_model_performance.eps`
- `plot9_sequence_length_vs_f1.eps`
- `plot10_video_sample_frames.eps`
- `plot11_video_loss.eps`
- `plot12_video_accuracy.eps`
- `plot13_video_confusion_matrix.eps`
- `plot14_seq2seq_loss.eps`

All plots are written using:

```python
fig.savefig(..., format="eps", dpi=600, bbox_inches="tight")
```

The numerical tables are saved as CSV files in:

```text
/content/experiment6_outputs/tables_csv/
```

and the complete output package is:

```text
/content/Experiment6_outputs.zip
```

The report should use the **actual values printed by your execution**, not values copied from another run.
